In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
print(os.environ.get('OPENAI_API_KEY')[:20])

sk-proj-tiADwcR4xi2q


In [2]:
# import v1.0
from langchain_openai.chat_models import ChatOpenAI
from langchain_core.prompts.prompt import PromptTemplate
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

# LLM 모델
# FewShotPromptTemplate
from langchain_core.prompts.few_shot import FewShotPromptTemplate

## FewShotPromptTemplate 설계

In [44]:
examples = [
    {
        "country": "프랑스",
        "question": "프랑스에 대해서 어떻게 알고 있나요?",
        "answer": "프랑스는 유럽 서부에 위치한 나라로, 파리가 수도입니다. 에펠탑과 루브르 박물관으로 유명하며 패션과 음식 문화가 발달해 있습니다."
    },
    {
        "country": "일본",
        "question": "일본에 대해서 어떻게 알고 있나요?",
        "answer": "일본은 동아시아에 위치한 섬나라로, 도쿄가 수도입니다. 애니메이션과 전자기기 산업이 유명하며 벚꽃 문화로도 잘 알려져 있습니다."
    },
    {
        "country": "캐나다",
        "question": "캐나다에 대해서 어떻게 알고 있나요?",
        "answer": "캐나다는 북아메리카에 위치한 나라로, 수도는 오타와입니다. 자연환경이 아름답고 다문화 사회로 유명하며 영어와 프랑스어를 함께 사용합니다."
    }
]

## 2단계: 예시용 프롬프트 템플릿 정의

In [45]:
examples_template = """
    Human: {country}
    AI: {answer}
"""

examples_prompt = PromptTemplate.from_template(examples_template)
examples_prompt

PromptTemplate(input_variables=['answer', 'country'], input_types={}, partial_variables={}, template='\n    Human: {country}\n    AI: {answer}\n')

## 3단계: FewShotPromptTemplate 생성 후 결합

In [46]:
prompt = FewShotPromptTemplate(
    # 예시
    examples=examples,

    # 예시를 어떤 형식으로 보여줄지
    example_prompt=examples_prompt,

    # 사용자 질문
    suffix="Human: {country}에 대해서 어떻게 알고 있어요?",
    input_variables=["country"]
)

In [47]:
prompt

FewShotPromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, examples=[{'country': '프랑스', 'question': '프랑스에 대해서 어떻게 알고 있나요?', 'answer': '프랑스는 유럽 서부에 위치한 나라로, 파리가 수도입니다. 에펠탑과 루브르 박물관으로 유명하며 패션과 음식 문화가 발달해 있습니다.'}, {'country': '일본', 'question': '일본에 대해서 어떻게 알고 있나요?', 'answer': '일본은 동아시아에 위치한 섬나라로, 도쿄가 수도입니다. 애니메이션과 전자기기 산업이 유명하며 벚꽃 문화로도 잘 알려져 있습니다.'}, {'country': '캐나다', 'question': '캐나다에 대해서 어떻게 알고 있나요?', 'answer': '캐나다는 북아메리카에 위치한 나라로, 수도는 오타와입니다. 자연환경이 아름답고 다문화 사회로 유명하며 영어와 프랑스어를 함께 사용합니다.'}], example_prompt=PromptTemplate(input_variables=['answer', 'country'], input_types={}, partial_variables={}, template='\n    Human: {country}\n    AI: {answer}\n'), suffix='Human: {country}에 대해서 어떻게 알고 있어요?')

In [48]:
chat = ChatOpenAI(temperature=0)

In [49]:
chain = prompt | chat

In [50]:
result = chain.invoke({"country": "한국"})

In [52]:
print(result.content)

AI: 한국은 동아시아에 위치한 나라로, 수도는 서울입니다. 한류와 K-pop으로 유명하며 전통문화와 현대문화가 조화롭게 공존하는 나라입니다. 또한 한식과 한국의 아름다운 자연경관도 많은 이들에게 사랑받고 있습니다.


In [16]:
# v1.0
# ChatModel
from langchain_core.prompts.few_shot import FewShotChatMessagePromptTemplate
from langchain_core.prompts.chat import ChatPromptTemplate

## FewShotPromptTemplate 설계

In [59]:
examples = [
    {
        "country": "프랑스",
        "answer": "프랑스는 유럽 서부에 위치한 나라로, 파리가 수도입니다. 에펠탑과 루브르 박물관으로 유명하며 패션과 음식 문화가 발달해 있습니다."
    },
    {
        "country": "일본",
        "answer": "일본은 동아시아에 위치한 섬나라로, 도쿄가 수도입니다. 애니메이션과 전자기기 산업이 유명하며 벚꽃 문화로도 잘 알려져 있습니다."
    },
    {
        "country": "캐나다",
        "answer": "캐나다는 북아메리카에 위치한 나라로, 수도는 오타와입니다. 자연환경이 아름답고 다문화 사회로 유명하며 영어와 프랑스어를 함께 사용합니다."
    }
]

## 2단계 예시용 프름프트 템플릿 정의

In [60]:
example = ChatPromptTemplate.from_messages([
    
    ("human", "{country}어떻게 알고있나요?"),
    ("ai", "{answer}")
    
])

In [61]:
example_prompt = FewShotChatMessagePromptTemplate(
    examples=examples,
    example_prompt=example
)

In [62]:
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 지리학 전문가입니다. 짧은 답변을 제공하세요."),
    example_prompt,
    ("human", "{country}어떻게 알고있나요?"),
])

In [63]:
print(final_prompt)

input_variables=['country'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='당신은 지리학 전문가입니다. 짧은 답변을 제공하세요.'), additional_kwargs={}), FewShotChatMessagePromptTemplate(examples=[{'country': '프랑스', 'answer': '프랑스는 유럽 서부에 위치한 나라로, 파리가 수도입니다. 에펠탑과 루브르 박물관으로 유명하며 패션과 음식 문화가 발달해 있습니다.'}, {'country': '일본', 'answer': '일본은 동아시아에 위치한 섬나라로, 도쿄가 수도입니다. 애니메이션과 전자기기 산업이 유명하며 벚꽃 문화로도 잘 알려져 있습니다.'}, {'country': '캐나다', 'answer': '캐나다는 북아메리카에 위치한 나라로, 수도는 오타와입니다. 자연환경이 아름답고 다문화 사회로 유명하며 영어와 프랑스어를 함께 사용합니다.'}], input_variables=[], input_types={}, partial_variables={}, example_prompt=ChatPromptTemplate(input_variables=['answer', 'country'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, template='{country}어떻게 알고있나요?'), additional_kwargs={}), AIMessagePromptTemplate(prompt=Prompt

In [43]:
chat = ChatOpenAI(temperature=0)

In [33]:
final_chat = final_prompt | chat

In [37]:
result = final_chat.invoke({"country": "짐바브웨"})

In [65]:
print(result.content)

AI: 한국은 동아시아에 위치한 나라로, 수도는 서울입니다. 한류와 K-pop으로 유명하며 전통문화와 현대문화가 조화롭게 공존하는 나라입니다. 또한 한식과 한국의 아름다운 자연경관도 많은 이들에게 사랑받고 있습니다.


## LengthBasedExampleSelector 설계

In [66]:
# v1.0
from langchain_core.example_selectors.length_based import LengthBasedExampleSelector

In [ ]:
examples = [
    {
        "country": "프랑스",
        "answer": "프랑스는 유럽 서부에 위치한 나라로, 파리가 수도입니다. 에펠탑과 루브르 박물관으로 유명하며 패션과 음식 문화가 발달해 있습니다."
    },
    {
        "country": "일본",
        "answer": "일본은 동아시아에 위치한 섬나라로, 도쿄가 수도입니다. 애니메이션과 전자기기 산업이 유명하며 벚꽃 문화로도 잘 알려져 있습니다."
    },
    {
        "country": "캐나다",
        "answer": "캐나다는 북아메리카에 위치한 나라로, 수도는 오타와입니다. 자연환경이 아름답고 다문화 사회로 유명하며 영어와 프랑스어를 함께 사용합니다."
    }
]

In [ ]:
example_prompt = PromptTemplate.from_template("human: {country}")

In [ ]:
example_selector = LengthBasedExampleSelector(
    examples=examples,
    example_prompt=example_prompt,
    # 예제의 토큰개수
    max_length=100
)

In [ ]:
prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,
    example_selector=example_selector,
    suffix="Human", "{country}어떻게 알고있나요?",
    input_variables=["country"]
)

In [ ]:
prompt.format(**{
    "country" : "브라질"
})

In [ ]:
final_chat = prompt | chat

In [ ]:
result = final_chat.invoke({"country": "짐바브웨"})

In [ ]:
print(result.content)

## Chat 모델(LengthBasedExampleSelector)

In [68]:
examples = [
    {
        "country": "프랑스",
        "answer": "프랑스는 유럽 서부에 위치한 나라로, 파리가 수도입니다. 에펠탑과 루브르 박물관으로 유명하며 패션과 음식 문화가 발달해 있습니다."
    },
    {
        "country": "일본",
        "answer": "일본은 동아시아에 위치한 섬나라로, 도쿄가 수도입니다. 애니메이션과 전자기기 산업이 유명하며 벚꽃 문화로도 잘 알려져 있습니다."
    },
    {
        "country": "캐나다",
        "answer": "캐나다는 북아메리카에 위치한 나라로, 수도는 오타와입니다. 자연환경이 아름답고 다문화 사회로 유명하며 영어와 프랑스어를 함께 사용합니다."
    }
]

In [69]:
length_prompt = PromptTemplate(
    input_variables=["country", "answer"],
    template="Human: {country} 어떻게 알고 있나요?\nAI: {answer}"
)

example_selector = LengthBasedExampleSelector(
    examples=examples,
    example_prompt=length_prompt,
    max_length=100
)

fewshot_prompt = FewShotPromptTemplate(
    example_prompt=length_prompt,
    example_selector=example_selector,
    suffix="Human: {country} 어떻게 알고 있나요?",
    input_variables=["country"]
)

In [71]:
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 지리학 전문가입니다. 짧은 답변을 제공하세요."),
    ("human", fewshot_prompt.format(country="{country}")),
    ("human", "{country} 어떻게 알고 있나요?"),
])

In [72]:
chain = final_prompt | chat

In [73]:
result = chain.invoke({"country": "짐바브웨"})

In [74]:
print(result.content)

짐바브웨는 아프리카 대륙 남부에 위치한 국가로, 하라레가 수도입니다. 빅토리아 폭포와 사바나 지역으로 유명하며, 자원 풍부한 국가로 알려져 있습니다.
